# Routing Agent Example

This notebook shows why LangGraph is useful.

The agent can choose between different paths:

- refund
- cook_food
- ask_question
- call_tool
- end_chat

## Easy Restaurant Example

Imagine a restaurant help desk.

Different customer messages need different actions:

| Customer says | Agent route |
| --- | --- |
| I want biryani | cook_food |
| I want my money back | refund |
| What is on the menu? | ask_question |
| Check my order status | call_tool |
| Bye | end_chat |

LangGraph lets us write these routes clearly.

In [ ]:
from typing import Annotated, TypedDict

import gradio as gr
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## Define State

State is the information bag.

It carries the user message, selected route, and final response.

In [ ]:
class RoutingState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    user_message: str
    route: str
    final_response: str

## Decision Node

This node reads the customer message and decides which route to follow.

For learning, we use simple keyword rules.

Later, this decision can be made by an LLM.

In [ ]:
def decide_route(state: RoutingState) -> RoutingState:
    user_message = state["user_message"].lower()

    if "refund" in user_message or "money back" in user_message:
        route = "refund"
    elif "status" in user_message or "track" in user_message or "check" in user_message:
        route = "call_tool"
    elif "?" in user_message or "available" in user_message or "menu" in user_message:
        route = "ask_question"
    elif "bye" in user_message or "exit" in user_message or "stop" in user_message:
        route = "end_chat"
    else:
        route = "cook_food"

    print("Selected route:", route)
    return {"route": route, "messages": [f"Router selected: {route}"]}

## Action Nodes

Each function below is one possible path.

Think of each node as a different counter in the restaurant.

In [ ]:
def refund_node(state: RoutingState) -> RoutingState:
    response = "I understand. I will start your refund request. Please share your order number."
    return {"final_response": response, "messages": [response]}


def cook_food_node(state: RoutingState) -> RoutingState:
    response = "Your order is accepted. The kitchen will start preparing your food now."
    return {"final_response": response, "messages": [response]}


def ask_question_node(state: RoutingState) -> RoutingState:
    response = "Sure. Please tell me which item you want, for example biryani, burger, or pizza."
    return {"final_response": response, "messages": [response]}


def call_tool_node(state: RoutingState) -> RoutingState:
    # Fake tool for learning. Later this could call a real database or API.
    fake_order_status = "Your order is being prepared and will be ready in 15 minutes."
    return {"final_response": fake_order_status, "messages": [fake_order_status]}


def end_chat_node(state: RoutingState) -> RoutingState:
    response = "Thank you. Chat ended. Have a nice day!"
    return {"final_response": response, "messages": [response]}

## Routing Function

This function tells LangGraph which edge to follow after the decision node.

In [ ]:
def route_after_decision(state: RoutingState) -> str:
    return state["route"]

## Build Conditional Graph

This is the important LangGraph part.

`add_conditional_edges` means: choose the next node based on the route.

In [ ]:
routing_builder = StateGraph(RoutingState)

routing_builder.add_node("decide_route", decide_route)
routing_builder.add_node("refund", refund_node)
routing_builder.add_node("cook_food", cook_food_node)
routing_builder.add_node("ask_question", ask_question_node)
routing_builder.add_node("call_tool", call_tool_node)
routing_builder.add_node("end_chat", end_chat_node)

routing_builder.add_edge(START, "decide_route")

routing_builder.add_conditional_edges(
    "decide_route",
    route_after_decision,
    {
        "refund": "refund",
        "cook_food": "cook_food",
        "ask_question": "ask_question",
        "call_tool": "call_tool",
        "end_chat": "end_chat",
    }
)

routing_builder.add_edge("refund", END)
routing_builder.add_edge("cook_food", END)
routing_builder.add_edge("ask_question", END)
routing_builder.add_edge("call_tool", END)
routing_builder.add_edge("end_chat", END)

routing_graph = routing_builder.compile()

## Display Graph

In [ ]:
routing_graph

## Test All Routes

Run this cell to see all five paths.

In [ ]:
examples = [
    "I want chicken biryani",
    "I want a refund",
    "What is available on the menu?",
    "Please check my order status",
    "bye"
]

for text in examples:
    print("User:", text)
    output = routing_graph.invoke({"user_message": text, "messages": []})
    print("Agent:", output["final_response"])
    print("-" * 50)

## Gradio App

This app lets you type your own message and see which route the agent uses.

## Example Output

When you run the test cell, you should see output like this:

```text
User: I want chicken biryani
Agent: Your order is accepted. The kitchen will start preparing your food now.
--------------------------------------------------
User: I want a refund
Agent: I understand. I will start your refund request. Please share your order number.
--------------------------------------------------
User: What is available on the menu?
Agent: Sure. Please tell me which item you want, for example biryani, burger, or pizza.
--------------------------------------------------
User: Please check my order status
Agent: Your order is being prepared and will be ready in 15 minutes.
--------------------------------------------------
User: bye
Agent: Thank you. Chat ended. Have a nice day!
```

In [ ]:
def routing_agent_app(user_message: str) -> str:
    if not user_message.strip():
        return "Please type a message."

    output = routing_graph.invoke({
        "user_message": user_message,
        "messages": []
    })

    return output["final_response"]

In [ ]:
routing_demo = gr.Interface(
    fn=routing_agent_app,
    inputs=gr.Textbox(label="Customer Message", placeholder="Try: I want a refund"),
    outputs=gr.Textbox(label="Agent Reply"),
    title="Restaurant Routing Agent",
    description="LangGraph chooses between refund, cook_food, ask_question, call_tool, and end_chat."
)

routing_demo.launch()